# ACE F2 — Reproduksi Adaptive Strategy Pilot

**Run all** untuk menjalankan 30 tes dan 6 run A/B, menyimpan log, dan mengunduh ACE_F2_reproduced_evidence.zip.

Eksperimen ini menguji seleksi strategi dari katalog algoritma yang telah ditulis manusia. Ini **bukan** agen AI yang menciptakan algoritma sendiri. Tidak memerlukan API, token, GPU atau paket berbayar.

Sistem Colab bersifat sementara. Simpan ZIP agar hasil tidak hilang. Held-out F4 tidak disertakan.


In [ ]:
# ACE F2: self-contained stdlib only; no model/API calls, no external downloads.
import base64, hashlib, json, pathlib, subprocess, sys, tempfile, zipfile
from pathlib import Path

embedded = {'ace_f1.py': '...', 'test_f1.py': 'IiIiVHdlbHZlIGFjY2VwdGFuY2Ugc2NlbmFyaW9zIGZyb20gQUNFIEYxIEJsdWVwcmludCDCpzEwICsgc3RyaWN0IHN1cHBsZW1lbnRhbCBjaGVja3MuIiIiCmltcG9ydCBjb3B5CmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaAppbXBvcnQgYWNlX2YxIGFzIGgKCmNsYXNzIEFjY2VwdGFuY2VUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgc2V0VXAoc2VsZik6CiAgICAgICAgc2VsZi50bXA9dGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkKICAgICAgICBzZWxmLnJvb3Q9UGF0aChzZWxmLnRtcC5uYW1lKQogICAgZGVmIHRlYXJEb3duKHNlbGYpOiBzZWxmLnRtcC5jbGVhbnVwKCkKCiAgICBkZWYgdGVzdF9GMV9UMDFfc3ltYm9saWNfcmVmZXJlbmNlX2VtcHR5X2R1cGxpY2F0ZShzZWxmKToKICAgICAgICBmb3IgdG9rZW5zIGluIChbXSxbJ2EnXSxbJ2EnLCdhJywnYicsJ2EnXSk6CiAgICAgICAgICAgIGZvciBvcGVyYXRpb24gaW4gW3snb3AnOidyZXZlcnNlJ30seydvcCc6J3JvdGF0ZV9sZWZ0Jywnc3RlcHMnOjEyfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgeydvcCc6J2ZpbHRlcl9lcScsJ3ZhbHVlJzonYSd9LHsnb3AnOidkZWR1cGUnfSx7J29wJzonc3dhcF9wYWlycyd9XToKICAgICAgICAgICAgICAgIHJlc3VsdD1oLmludGVycHJldF9zKHsndG9rZW5zJzp0b2tlbnMsJ29wZXJhdGlvbnMnOltvcGVyYXRpb25dfSkKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0SXNJbnN0YW5jZShyZXN1bHQsbGlzdCkKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0LGguaW50ZXJwcmV0X3Moeyd0b2tlbnMnOnRva2Vucywnb3BlcmF0aW9ucyc6W29wZXJhdGlvbl19KSkKICAgICAgICB0YXNrcz1oLmdlbmVyYXRlKGguU0VFRFNbMF0pOyBzZWxmLmFzc2VydEVxdWFsKGxlbih0YXNrcyksMzYpCiAgICAgICAgdD10YXNrc1swXTtzZWxmLmFzc2VydEVxdWFsKGguZXZhbHVhdGUodCxoLmFfZml4ZWRfc29sdmUoaC5wdWJsaWNfdGFzayh0KSkpWydzY29yZSddLDEpCgogICAgZGVmIHRlc3RfRjFfVDAyX2dyYXBoX29yYWNsZV90aWVzX3VucmVhY2hhYmxlX3plcm8oc2VsZik6CiAgICAgICAgdD1uZXh0KHggZm9yIHggaW4gaC5nZW5lcmF0ZShoLlNFRURTWzBdKSBpZiB4WydmYW1pbHknXT09J0cnKQogICAgICAgIHQ9Y29weS5kZWVwY29weSh0KQogICAgICAgIHA9eydraW5kJzonZ3JhcGgnLCdub2Rlcyc6WydBJywnQicsJ0MnLCdEJ10sCiAgICAgICAgICAgJ2VkZ2VzJzpbWydBJywnQicsMF0sWydBJywnQycsMF0sWydCJywnRCcsMl0sWydDJywnRCcsMl1dLAogICAgICAgICAgICdkaXJlY3RlZCc6RmFsc2UsJ3N0YXJ0JzonQScsJ2dvYWwnOidEJ30KICAgICAgICB0Wydwcm9ibGVtJ109cAogICAgICAgIGZvciBjYW5kaWRhdGUgaW4gW1snQScsJ0InLCdEJ10sWydBJywnQycsJ0QnXV06CiAgICAgICAgICAgIG89eyd0YXNrX2lkJzp0Wyd0YXNrX2lkJ10sJ2Fuc3dlcic6eydwYXRoJzpjYW5kaWRhdGUsJ2Nvc3QnOjIsJ3JlYWNoYWJsZSc6VHJ1ZX19CiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoaC5ldmFsdWF0ZSh0LG8pWydzdGF0dXMnXSwnUEFTUycpCiAgICAgICAgcFsnZWRnZXMnXT1bXQogICAgICAgIG89eyd0YXNrX2lkJzp0Wyd0YXNrX2lkJ10sJ2Fuc3dlcic6eydwYXRoJzpbXSwnY29zdCc6Tm9uZSwncmVhY2hhYmxlJzpGYWxzZX19CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChoLmV2YWx1YXRlKHQsbylbJ3N0YXR1cyddLCdQQVNTJykKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKGgub3JhY2xlX2dyYXBoKHApWydyZWFjaGFibGUnXSkKCiAgICBkZWYgdGVzdF9GMV9UMDNfaW52YWxpZF9vdXRwdXRfc2NoZW1hcyhzZWxmKToKICAgICAgICB0PWguZ2VuZXJhdGUoaC5TRUVEU1swXSlbMF0KICAgICAgICBmb3IgbyBpbiAoeydhbnN3ZXInOnsndG9rZW5zJzpbXX19LCB7J3Rhc2tfaWQnOnRbJ3Rhc2tfaWQnXX0sCiAgICAgICAgICAgICAgICAgIHsndGFza19pZCc6dFsndGFza19pZCddLCdhbnN3ZXInOnsndG9rZW5zJzonYSd9fSwKICAgICAgICAgICAgICAgICAgeyd0YXNrX2lkJzp0Wyd0YXNrX2lkJ10sJ2Fuc3dlcic6eyd0b2tlbnMnOlsxXX19LAogICAgICAgICAgICAgICAgICB7J3Rhc2tfaWQnOnRbJ3Rhc2tfaWQnXSwnYW5zd2VyJzp7J3Rva2Vucyc6W10sJ2V4dHJhJzoxfX0pOgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGguZXZhbHVhdGUodCxvKVsnc3RhdHVzJ10sJ0lOVkFMSURfT1VUUFVUJykKCiAgICBkZWYgdGVzdF9GMV9UMDRfcmVwZWF0YWJpbGl0eV92ZXJzaW9uc19zZWVkcyhzZWxmKToKICAgICAgICBmb3Igc2VlZCBpbiBoLlNFRURTOgogICAgICAgICAgICB0MT1oLmdlbmVyYXRlKHNlZWQpO3QyPWguZ2VuZXJhdGUoc2VlZCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChoLmRpZ2VzdCh0MSksaC5kaWdlc3QodDIpKQogICAgICAgICAgICB4MT1baC5hX2ZpeGVkX3NvbHZlKGgucHVibGljX3Rhc2sodCkpIGZvciB0IGluIHQxXQogICAgICAgICAgICB4Mj1baC5hX2ZpeGVkX3NvbHZlKGgucHVibGljX3Rhc2sodCkpIGZvciB0IGluIHQyXQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGguZGlnZXN0KHgxKSxoLmRpZ2VzdCh4MikpCiAgICAgICAgc2VsZi5hc3NlcnROb3RFcXVhbChoLmRpZ2VzdChoLmdlbmVyYXRlKGguU0VFRFNbMF0pKSxoLmRpZ2VzdChoLmdlbmVyYXRlKGguU0VFRFNbMV0pKSkKCiAgICBkZWYgdGVzdF9GMV9UMDVfYmFzZWxpbmVfZnJvemVuX2FuZF9ub19sZWFybihzZWxmKToKICAgICAgICBiZWZvcmU9aC5wcm92ZW5hbmNlKClbJ2NvZGVfc2hhMjU2J10KICAgICAgICB0YXNrcz1oLmdlbmVyYXRlKGguU0VFRFNbMF0pO3Q9bmV4dCh4IGZvciB4IGluIHRhc2tzIGlmIHhbJ2ZhbWlseSddPT0nUycpCiAgICAgICAgYT1oLmFfZml4ZWRfc29sdmUoaC5wdWJsaWNfdGFzayh0KSkKICAgICAgICBmb3Igb3RoZXIgaW4gdGFza3M6CiAgICAgICAgICAgIGguYV9maXhlZF9zb2x2ZShoLnB1YmxpY190YXNrKG90aGVyKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGEsaC5hX2ZpeGVkX3NvbHZlKGgucHVibGljX3Rhc2sodCkpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYmVmb3JlLGgucHJvdmVuYW5jZSgpWydjb2RlX3NoYTI1NiddKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoYW55KGsgaW4gaC5wdWJsaWNfdGFzayh0KSBmb3IgayBpbiAoJ3NwbGl0Jywnc2VlZCcsJ3RlbXBsYXRlX2lkJykpKQoKICAgIGRlZiB0ZXN0X0YxX1QwNl9oZWxkb3V0X2FwaV9kZW5pYWxfbm9fbWF0ZXJpYWwoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhoLkFjY2Vzc0RlbmllZCk6aC5nZXRfdGFzaygnaGVsZC1vdXQnLGguZ2VuZXJhdGUoaC5TRUVEU1swXSkpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbCh0WydzcGxpdCddIT0naGVsZC1vdXQnIGZvciB0IGluIGguZ2VuZXJhdGUoaC5TRUVEU1swXSkpKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoaC5BY2Nlc3NEZW5pZWQpOgogICAgICAgICAgICBoLnZhbGlkYXRlX3Rhc2tfbWFuaWZlc3QoW3sndGFza19pZCc6J0ctSE8tMDEnLCdmYW1pbHknOidHJywnc3BsaXQnOidoZWxkLW91dCcsJ3Byb2JsZW0nOnt9fV0pCgogICAgZGVmIHRlc3RfRjFfVDA3X2NyYXNoX2FmdGVyX2V2ZW50X2JlZm9yZV9jaGVja3BvaW50X2lkZW1wb3RlbnRfcmVzdW1lKHNlbGYpOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoSW50ZXJydXB0ZWRFcnJvcik6CiAgICAgICAgICAgIGgucnVuKGguU0VFRFNbMF0sc2VsZi5yb290LHNpbXVsYXRlX3N0YWxlX2NoZWNrcG9pbnQ9VHJ1ZSkKICAgICAgICBydW5fZGlyPXNlbGYucm9vdC8ncnVucycvZidBX0ZJWEVEX3toLlNFRURTWzBdfScKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihoLnJlYWRfbGVkZ2VyKHJ1bl9kaXIvJ2V2ZW50cy5qc29ubCcpKSwxKQogICAgICAgIHN0YXRlPWgucnVuKGguU0VFRFNbMF0sc2VsZi5yb290KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3RhdGVbJ3Rhc2tfcmVzdWx0cyddLDM2KQogICAgICAgIHJvd3M9aC5ldmVudF9yZXN1bHRzKGgucmVhZF9sZWRnZXIocnVuX2Rpci8nZXZlbnRzLmpzb25sJykpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4ocm93cyksbGVuKHtyWyd0YXNrX2lkJ10gZm9yIHIgaW4gcm93c30pKQogICAgICAgIGJlZm9yZT0ocnVuX2Rpci8nZXZlbnRzLmpzb25sJykucmVhZF9ieXRlcygpCiAgICAgICAgaC5ydW4oaC5TRUVEU1swXSxzZWxmLnJvb3QpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChiZWZvcmUsKHJ1bl9kaXIvJ2V2ZW50cy5qc29ubCcpLnJlYWRfYnl0ZXMoKSkKCiAgICBkZWYgdGVzdF9GMV9UMDhfYnVkZ2V0X2FuZF93YWxsX2xpbWl0cyhzZWxmKToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKGguQnVkZ2V0RXJyb3IpOmgucnVuKGguU0VFRFNbMF0sc2VsZi5yb290LG1heF9jb3N0X2lkcj0xKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoKHNlbGYucm9vdC8ncnVucycpLmV4aXN0cygpKQogICAgICAgIHdpdGggcGF0Y2gub2JqZWN0KGgsJ1JVTl9USU1FT1VUX1NFQ09ORFMnLC0xKToKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhoLkJ1ZGdldEVycm9yKTpoLnJ1bihoLlNFRURTWzBdLHNlbGYucm9vdCkKICAgICAgICBvdXQ9c2VsZi5yb290LydydW5zJy9mJ0FfRklYRURfe2guU0VFRFNbMF19JwogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoKG91dC8nZXZlbnRzLmpzb25sJykuZXhpc3RzKCkpCgogICAgZGVmIHRlc3RfRjFfVDA5X2ludmFsaWRfbWFuaWZlc3RfYW5kX2V2YWx1YXRvcl9lcnJvcnNfZmFpbF9jbG9zZWQoc2VsZik6CiAgICAgICAgdGFza3M9aC5nZW5lcmF0ZShoLlNFRURTWzBdKQogICAgICAgIGJhZD1jb3B5LmRlZXBjb3B5KHRhc2tzKTtiYWRbMF1bJ3Rhc2tfaWQnXT1iYWRbMV1bJ3Rhc2tfaWQnXQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6aC52YWxpZGF0ZV90YXNrX21hbmlmZXN0KGJhZCkKICAgICAgICBiYWQ9Y29weS5kZWVwY29weSh0YXNrcyk7YmFkWzBdWyd0ZW1wbGF0ZV9pZCddPWJhZFsxMl1bJ3RlbXBsYXRlX2lkJ10KICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOmgudmFsaWRhdGVfdGFza19tYW5pZmVzdChiYWQpCiAgICAgICAgYmFkPWNvcHkuZGVlcGNvcHkodGFza3NbMF0pO2JhZFsncHJvYmxlbSddWydvcGVyYXRpb25zJ109W3snb3AnOidVTlJFQ09HTklaRUQnfV0KICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOmguZXZhbHVhdGUoYmFkLHsndGFza19pZCc6YmFkWyd0YXNrX2lkJ10sJ2Fuc3dlcic6eyd0b2tlbnMnOltdfX0pCgogICAgZGVmIHRlc3RfRjFfVDEwX2V2ZW50X2FuZF9yZXN1bHRfdGFtcGVyaW5nX2RldGVjdGVkKHNlbGYpOgogICAgICAgIGgucnVuKGguU0VFRFNbMF0sc2VsZi5yb290KQogICAgICAgIGZvbGRlcj1zZWxmLnJvb3QvJ3J1bnMnL2YnQV9GSVhFRF97aC5TRUVEU1swXX0nCiAgICAgICAgZj1mb2xkZXIvJ2V2ZW50cy5qc29ubCc7b3JpZz1mLnJlYWRfYnl0ZXMoKQogICAgICAgIGYud3JpdGVfYnl0ZXMob3JpZy5yZXBsYWNlKGInQV9GSVhFRCcsYidCX0ZJWEVEJywxKSkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKGguSW50ZWdyaXR5RXJyb3IpOmgudmVyaWZ5X3J1bihmb2xkZXIpCiAgICAgICAgZi53cml0ZV9ieXRlcyhvcmlnKQogICAgICAgIHA9Zm9sZGVyLyd0YXNrX3Jlc3VsdHMuanNvbmwnO3RleHQ9cC5yZWFkX3RleHQoKTtwLndyaXRlX3RleHQodGV4dC5yZXBsYWNlKCdQQVNTJywnRkFLRScsMSkpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhoLkludGVncml0eUVycm9yKTpoLnZlcmlmeV9ydW4oZm9sZGVyKQoKICAgIGRlZiB0ZXN0X0YxX1QxMV9mdWxsX3JlY29uY2lsaWF0aW9uX2FsbF9zZWVkcyhzZWxmKToKICAgICAgICBmb3Igc2VlZCBpbiBoLlNFRURTOmgucnVuKHNlZWQsc2VsZi5yb290KQogICAgICAgIHJlcD1oLnN1bW1hcml6ZShzZWxmLnJvb3QpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXBbJ3RvdGFsX3Rhc2tfYXR0ZW1wdHMnXSwxMDgpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXBbJ2hlbGRfb3V0X2F0dGVtcHRzJ10sMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bSh4Wyd0b3RhbCddIGZvciB4IGluIHJlcFsncGVyX2ZhbWlseV9zcGxpdCddLnZhbHVlcygpKSwxMDgpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXBbJ3RvdGFsX3Bhc3MnXSxzdW0oeFsncGFzcyddIGZvciB4IGluIHJlcFsncGVyX2ZhbWlseV9zcGxpdCddLnZhbHVlcygpKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihyZXBbJ3J1bnMnXSksMykKICAgICAgICBzZWxmLmFzc2VydFRydWUoYWxsKHhbJ2NvbXBsZXRlJ10gZm9yIHggaW4gcmVwWydydW5zJ10pKQoKICAgIGRlZiB0ZXN0X0YxX1QxMl9zbmFwc2hvdF9leHBvcnRfcG9ydGFiaWxpdHlfYW5kX3NyY19oYXNoKHNlbGYpOgogICAgICAgIGgucnVuKGguU0VFRFNbMF0sc2VsZi5yb290KQogICAgICAgIGZvbGRlcj1zZWxmLnJvb3QvJ3J1bnMnL2YnQV9GSVhFRF97aC5TRUVEU1swXX0nCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGgudmVyaWZ5X3J1bihmb2xkZXIscmVxdWlyZV9jb21wbGV0ZT1UcnVlKVsnY29tcGxldGUnXSkKICAgICAgICBkZXN0aW5hdGlvbj1zZWxmLnJvb3QvJ21pZ3JhdGVkJwogICAgICAgIHNodXRpbC5jb3B5dHJlZShmb2xkZXIsZGVzdGluYXRpb24pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChoLnZlcmlmeV9ydW4oZm9sZGVyKSxoLnZlcmlmeV9ydW4oZGVzdGluYXRpb24pKQogICAgICAgIGNmZz1oLnJlYWRfanNvbihmb2xkZXIvJ2NvbmZpZy5zbmFwc2hvdC5qc29uJykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGNmZ1sncHJvdmVuYW5jZSddWydjb2RlX3NoYTI1NiddLGguZmlsZV9oYXNoKFBhdGgoaC5fX2ZpbGVfXykpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoY2ZnWydwcm92ZW5hbmNlJ11bJ2FkZGl0aW9uYWxfY29zdF9pZHInXSwwKQoKICAgIGRlZiB0ZXN0X3N1cHBsZW1lbnRfaGVsZG91dF90eXBlX21pc21hdGNoX2RlbmlhbChzZWxmKToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOmguZ2V0X3Rhc2soJ2ZvbycsW10pCiAgICAgICAgdD1oLmdlbmVyYXRlKGguU0VFRFNbMF0pWzBdCiAgICAgICAgYmFkPWgucHVibGljX3Rhc2sodCk7YmFkWydzZWVkJ109MAogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6aC5hX2ZpeGVkX3NvbHZlKGJhZCkKCiAgICBkZWYgdGVzdF9zdXBwbGVtZW50X3Jlc3BvbnNlX2Nvc3RfcmVjb21wdXRlZF9hbmRfY29uc3RyYWludHMoc2VsZik6CiAgICAgICAgdD1uZXh0KHggZm9yIHggaW4gaC5nZW5lcmF0ZShoLlNFRURTWzBdKSBpZiB4Wyd0ZW1wbGF0ZV9pZCddPT0nRy5tdXN0X3Zpc2l0LnYxJykKICAgICAgICBiYWQ9aC5hX2ZpeGVkX3NvbHZlKGgucHVibGljX3Rhc2sodCkpO2JhZFsnYW5zd2VyJ11bJ2Nvc3QnXT05OTk5CiAgICAgICAgc2VsZi5hc3NlcnROb3RFcXVhbChoLmV2YWx1YXRlKHQsYmFkKVsnc3RhdHVzJ10sJ1BBU1MnKQogICAgICAgIHE9Y29weS5kZWVwY29weSh0KQogICAgICAgIHFbJ3Byb2JsZW0nXVsnbWF4X2hvcHMnXT0xCiAgICAgICAgcVsncHJvYmxlbSddLnBvcCgnbXVzdF92aXNpdCcpCiAgICAgICAgeD17J3Rhc2tfaWQnOnFbJ3Rhc2tfaWQnXSwnYW5zd2VyJzp7J3BhdGgnOlsnQScsJ0InLCdFJ10sJ2Nvc3QnOjIsJ3JlYWNoYWJsZSc6VHJ1ZX19CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChoLmV2YWx1YXRlKHEseClbJ3N0YXR1cyddLCdJTkNPUlJFQ1QnKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6dW5pdHRlc3QubWFpbih2ZXJib3NpdHk9MikK', 'ace_f2.py': '...', 'test_f2.py': 'IiIiRjIgcGlsb3QgYWNjZXB0YW5jZSBjaGVja3M7IGluZGVwZW5kZW50IGZyb20gdGhlIEYxIHN1aXRlLiIiIgppbXBvcnQgY29weQppbXBvcnQganNvbgppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHVuaXR0ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHVuaXR0ZXN0Lm1vY2sgaW1wb3J0IHBhdGNoCmltcG9ydCBhY2VfZjEgYXMgZjEKaW1wb3J0IGFjZV9mMiBhcyBmMgoKY2xhc3MgRjJUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgc2V0VXAoc2VsZik6CiAgICAgICAgc2VsZi50PXRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpCiAgICAgICAgc2VsZi5yb290PVBhdGgoc2VsZi50Lm5hbWUpCiAgICBkZWYgdGVhckRvd24oc2VsZik6c2VsZi50LmNsZWFudXAoKQoKICAgIGRlZiB0ZXN0XzAxX2YxX2NvZGVfaXNfaWRlbnRpY2FsX3RvX3ZlcmlmaWVkX2Jhc2VsaW5lKHNlbGYpOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZjIuZnJvemVuX2ZpbGVzKClbJ2YxX3NoYTI1NiddLAogICAgICAgICAgICAgICAgICAgICAgICAgJzRlZjhhNWMwNmI3NWE5ZGYwMDM2N2NlNzQyMjY0NDNmMWMzOTRiMjlkMjUzZDNmOTc3MTgzZjhhOWRmM2E0NmInKQoKICAgIGRlZiB0ZXN0XzAyX3NlZWRfbWFuaWZlc3RfZGlzam9pbnRfYW5kX25vX2hlbGRvdXQoc2VsZik6CiAgICAgICAgZm9yIHMgaW4gZjIuU0VFRFM6CiAgICAgICAgICAgIHByb2JlLGZpbmFsPWYyLm1ha2VfbWFuaWZlc3QocykKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgobGVuKHByb2JlKSxsZW4oZmluYWwpKSwoNCwzMikpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KHRbJ3Rhc2tfaWQnXSBmb3IgdCBpbiBwcm9iZSkmc2V0KHRbJ3Rhc2tfaWQnXSBmb3IgdCBpbiBmaW5hbCksc2V0KCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwodFsnc3BsaXQnXSE9J2hlbGQtb3V0JyBmb3IgdCBpbiBwcm9iZStmaW5hbCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3VtKHRbJ3NwbGl0J109PSd2YWxpZGF0aW9uJyBmb3IgdCBpbiBmaW5hbCksOCkKCiAgICBkZWYgdGVzdF8wM19mdWxsX2hhbmRjcmFmdGVkX3N5bWJvbGljX21vZHVsZShzZWxmKToKICAgICAgICBmb3IgcyBpbiBmMi5TRUVEUzoKICAgICAgICAgICAgZm9yIHQgaW4gZjEuZ2VuZXJhdGUocyk6CiAgICAgICAgICAgICAgICBpZiB0WydmYW1pbHknXT09J1MnOgogICAgICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZjEuZXZhbHVhdGUodCxmMi5zb2x2ZShmMS5wdWJsaWNfdGFzayh0KSwnU19FWFRFTkRFRCcpKVsnc2NvcmUnXSwxKQoKICAgIGRlZiB0ZXN0XzA0X2dyYXBoX3NvbHZlcl9pbmRlcGVuZGVudF9jb3JyZWN0bmVzcyhzZWxmKToKICAgICAgICBmb3IgcyBpbiBmMi5TRUVEUzoKICAgICAgICAgICAgZm9yIHQgaW4gZjEuZ2VuZXJhdGUocyk6CiAgICAgICAgICAgICAgICBpZiB0WydmYW1pbHknXT09J0cnOgogICAgICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZjEuZXZhbHVhdGUodCxmMi5zb2x2ZShmMS5wdWJsaWNfdGFzayh0KSwnR19DT05TVFJBSU5FRCcpKVsnc2NvcmUnXSwxKQoKICAgIGRlZiB0ZXN0XzA1X2JlZ2luc193aXRoX2Zyb3plbl9hKHNlbGYpOgogICAgICAgIGZvciBzIGluIGYyLlNFRURTOgogICAgICAgICAgICBmb3IgdCBpbiBmMS5nZW5lcmF0ZShzKToKICAgICAgICAgICAgICAgIGlmIHRbJ2ZhbWlseSddPT0nUyc6IGFybT0nU19CQVNJQycKICAgICAgICAgICAgICAgIGVsc2U6YXJtPSdHX0JBU0lDJwogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmMi5zb2x2ZShmMS5wdWJsaWNfdGFzayh0KSxhcm0pLGYxLmFfZml4ZWRfc29sdmUoZjEucHVibGljX3Rhc2sodCkpKQoKICAgIGRlZiB0ZXN0XzA2X2NhdGFsb2d1ZV9hbmRfc3RyYXRlZ3lfc2xvdHMoc2VsZik6CiAgICAgICAgZm9yIG1vZGUgaW4gKCdBX0NPTlRST0wnLCdCX0FEQVBUJyk6CiAgICAgICAgICAgIF8sXyx0cmlhbHM9ZjIucGxhbihmMi5TRUVEU1swXSxtb2RlKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbih0cmlhbHMpLDEyKQogICAgICAgICAgICBpZiBtb2RlPT0nQV9DT05UUk9MJzoKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoYXJtLmVuZHN3aXRoKCdCQVNJQycpIGZvciBfLGFybSxfIGluIHRyaWFscykpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNldCh0Wyd0YXNrX2lkJ10gZm9yIHQsXyxfIGluIHRyaWFscykpLDQpCgogICAgZGVmIHRlc3RfMDdfYnVkZ2V0X3JlZnVzYWwoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhmMi5GMkJ1ZGdldEVycm9yKToKICAgICAgICAgICAgZjIucnVuKHNlbGYucm9vdCwnQl9BREFQVCcsZjIuU0VFRFNbMF0sbWF4X2Nvc3RfaWRyPTEpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZSgoc2VsZi5yb290LydydW5zJykuZXhpc3RzKCkpCgogICAgZGVmIHRlc3RfMDhfaW52YWxpZF9tb2RlbF9zZWVkX2FuZF9wdWJsaWNfb25seShzZWxmKToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOmYyLmNvbmYoJ0JfQURBUFQnLDk5OSkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOmYyLmNvbmYoJ0NfRVZPTFZFJyxmMi5TRUVEU1swXSkKICAgICAgICB0YXNrPWYxLnB1YmxpY190YXNrKGYxLmdlbmVyYXRlKGYyLlNFRURTWzBdKVswXSk7dGFza1snc3BsaXQnXT0naGVsZC1vdXQnCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTpmMi5zb2x2ZSh0YXNrLCdTX0VYVEVOREVEJykKCiAgICBkZWYgdGVzdF8wOV9jcmFzaF9yZXN1bWVfbm9fZHVwbGljYXRlKHNlbGYpOgogICAgICAgIHM9ZjIuU0VFRFNbMF0KICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKEludGVycnVwdGVkRXJyb3IpOgogICAgICAgICAgICBmMi5ydW4oc2VsZi5yb290LCdCX0FEQVBUJyxzLHNpbXVsYXRlX2ludGVycnVwdD1UcnVlKQogICAgICAgIGU9c2VsZi5yb290LydydW5zJy9mMi5ydW5faWQoJ0JfQURBUFQnLHMpLydldmVudHMuanNvbmwnCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oZjIucmVhZF9ldmVudHMoZSkpLDEpCiAgICAgICAgcmVzdWx0PWYyLnJ1bihzZWxmLnJvb3QsJ0JfQURBUFQnLHMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbJ3Byb2JlX2NvdW50J10sMTIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbJ2ZpbmFsX2NvdW50J10sMzIpCiAgICAgICAgYmVmb3JlPWUucmVhZF9ieXRlcygpO2YyLnJ1bihzZWxmLnJvb3QsJ0JfQURBUFQnLHMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlLnJlYWRfYnl0ZXMoKSxiZWZvcmUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oZjIucmVhZF9ldmVudHMoZSkpLDQ1KQoKICAgIGRlZiB0ZXN0XzEwX2N1dF9hZnRlcl9zb21lX2ZpbmFsX2FuZF9yZXN1bWUoc2VsZik6CiAgICAgICAgcz1mMi5TRUVEU1swXTttb2RlPSdCX0FEQVBUJwogICAgICAgIHBhcnQ9ZjIucnVuKHNlbGYucm9vdCxtb2RlLHMsc3RvcF9hZnRlcl9ldmVudHM9MTcpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShwYXJ0Wydjb21wbGV0ZSddKTtzZWxmLmFzc2VydEVxdWFsKHBhcnRbJ2ZpbmFsX2NvdW50J10sNCkKICAgICAgICBkb25lPWYyLnJ1bihzZWxmLnJvb3QsbW9kZSxzKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShkb25lWydjb21wbGV0ZSddKTtzZWxmLmFzc2VydEVxdWFsKGRvbmVbJ2ZpbmFsX2NvdW50J10sMzIpCgogICAgZGVmIHRlc3RfMTFfdGFtcGVyX2hhc2hfZGV0ZWN0ZWQoc2VsZik6CiAgICAgICAgcz1mMi5TRUVEU1swXTttb2RlPSdCX0FEQVBUJwogICAgICAgIGYyLnJ1bihzZWxmLnJvb3QsbW9kZSxzKQogICAgICAgIGQ9c2VsZi5yb290LydydW5zJy9mMi5ydW5faWQobW9kZSxzKTtwPWQvJ2V2ZW50cy5qc29ubCcKICAgICAgICBvcmlnPXAucmVhZF9ieXRlcygpO3Aud3JpdGVfYnl0ZXMob3JpZy5yZXBsYWNlKGInQl9BREFQVCcsYidDX0FEQVBUJywxKSkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKGYyLkYySW50ZWdyaXR5RXJyb3IpOmYyLnZlcmlmeV9ydW4oc2VsZi5yb290LG1vZGUscykKCiAgICBkZWYgdGVzdF8xMl90YW1wZXJfdmFsaWRfY2hhaW5fYnV0X2JhZF9zY29yZV9kZXRlY3RlZChzZWxmKToKICAgICAgICBzPWYyLlNFRURTWzBdO21vZGU9J0JfQURBUFQnCiAgICAgICAgZjIucnVuKHNlbGYucm9vdCxtb2RlLHMpCiAgICAgICAgZD1zZWxmLnJvb3QvJ3J1bnMnL2YyLnJ1bl9pZChtb2RlLHMpO3A9ZC8nZXZlbnRzLmpzb25sJwogICAgICAgIGV2PWYyLnJlYWRfZXZlbnRzKHApCiAgICAgICAgZXZbMF1bJ3Njb3JlJ109MS1ldlswXVsnc2NvcmUnXQogICAgICAgIHByZXY9JzAnKjY0CiAgICAgICAgZm9yIGUgaW4gZXY6CiAgICAgICAgICAgIGVbJ3ByZXZpb3VzX2hhc2gnXT1wcmV2O2VbJ2V2ZW50X2hhc2gnXT1mMi5zaGEoe2s6diBmb3Igayx2IGluIGUuaXRlbXMoKSBpZiBrIT0nZXZlbnRfaGFzaCd9KTtwcmV2PWVbJ2V2ZW50X2hhc2gnXQogICAgICAgIHAud3JpdGVfdGV4dCgnXG4nLmpvaW4oZjEuY2Fub25pY2FsKHgpIGZvciB4IGluIGV2KSsnXG4nKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoZjIuRjJJbnRlZ3JpdHlFcnJvcik6ZjIudmVyaWZ5X3J1bihzZWxmLnJvb3QsbW9kZSxzKQoKICAgIGRlZiB0ZXN0XzEzX3NlbGVjdGVkX3N0cmF0ZWd5X29ubHlfZnJvbV9wcm9iZXMoc2VsZik6CiAgICAgICAgcz1mMi5TRUVEU1swXTttb2RlPSdCX0FEQVBUJztmMi5ydW4oc2VsZi5yb290LG1vZGUscykKICAgICAgICBkPXNlbGYucm9vdC8ncnVucycvZjIucnVuX2lkKG1vZGUscykKICAgICAgICBldj1mMi5yZWFkX2V2ZW50cyhkLydldmVudHMuanNvbmwnKTtmcmVlemU9bmV4dCh4IGZvciB4IGluIGV2IGlmIHhbJ2FjdGlvbiddPT0nRlJFRVpFJykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGZyZWV6ZVsnc2VsZWN0ZWQnXSxmMi5zZWxlY3QoW3ggZm9yIHggaW4gZXYgaWYgeFsnYWN0aW9uJ109PSdQUk9CRSddLG1vZGUpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZnJlZXplWydldmVudF9pZCddLDEzKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoZVsnZXZlbnRfaWQnXT4xMyBmb3IgZSBpbiBldiBpZiBlWydhY3Rpb24nXT09J0ZJTkFMJykpCgogICAgZGVmIHRlc3RfMTRfZnVsbF9wYXJhbGxlbF9jb21wYXJpc29uKHNlbGYpOgogICAgICAgIGZvciBtb2RlIGluICgnQV9DT05UUk9MJywnQl9BREFQVCcpOgogICAgICAgICAgICBmb3IgcyBpbiBmMi5TRUVEUzpmMi5ydW4oc2VsZi5yb290LG1vZGUscykKICAgICAgICByZXBvcnQ9ZjIuc3VtbWFyeShzZWxmLnJvb3QpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4ocmVwb3J0WydydW5zJ10pLDYpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXBvcnRbJ2hlbGRvdXRfYXR0ZW1wdHMnXSwwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVwb3J0WydhZGRpdGlvbmFsX2Nvc3RfaWRyJ10sMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bSh2Wyd0b3RhbCddIGZvciB2IGluIHJlcG9ydFsnbWV0cmljcyddLnZhbHVlcygpKSwxOTIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW0odlsncGFzcyddIGZvciBrLHYgaW4gcmVwb3J0WydtZXRyaWNzJ10uaXRlbXMoKSBpZiBrLnN0YXJ0c3dpdGgoJ0FfQ09OVFJPTCcpKSw3OSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bSh2WydwYXNzJ10gZm9yIGssdiBpbiByZXBvcnRbJ21ldHJpY3MnXS5pdGVtcygpIGlmIGsuc3RhcnRzd2l0aCgnQl9BREFQVCcpKSw5NCkKCiAgICBkZWYgdGVzdF8xNV9jaGVja3BvaW50X2NvcnJ1cHRpb25fZGV0ZWN0ZWQoc2VsZik6CiAgICAgICAgcz1mMi5TRUVEU1swXTttb2RlPSdBX0NPTlRST0wnO2YyLnJ1bihzZWxmLnJvb3QsbW9kZSxzKQogICAgICAgIHA9c2VsZi5yb290LydydW5zJy9mMi5ydW5faWQobW9kZSxzKS8nY2hlY2twb2ludC5qc29uJwogICAgICAgIG9iaj1mMi5yZWFkX2pzb24ocCk7b2JqWydsZWRnZXJfc2hhMjU2J109J2YnKjY0O2YyLndyaXRlX2pzb24ocCxvYmopCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhmMi5GMkludGVncml0eUVycm9yKTpmMi52ZXJpZnlfcnVuKHNlbGYucm9vdCxtb2RlLHMpCgogICAgZGVmIHRlc3RfMTZfdGltZV9idWRnZXRfZW5mb3JjZWQoc2VsZik6CiAgICAgICAgd2l0aCBwYXRjaC5vYmplY3QoZjIsJ1RJTUVfQ0FQX1NFQ09ORFMnLC0xKToKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhmMi5GMkJ1ZGdldEVycm9yKToKICAgICAgICAgICAgICAgIGYyLnJ1bihzZWxmLnJvb3QsJ0JfQURBUFQnLGYyLlNFRURTWzBdKQogICAgICAgIGQ9c2VsZi5yb290LydydW5zJy9mMi5ydW5faWQoJ0JfQURBUFQnLGYyLlNFRURTWzBdKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoKGQvJ2V2ZW50cy5qc29ubCcpLmV4aXN0cygpKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6dW5pdHRlc3QubWFpbih2ZXJib3NpdHk9MikK'}
root = Path(tempfile.mkdtemp(prefix='ACE_F2_COLAB_'))
for name, payload in embedded.items():
    (root/name).write_bytes(base64.b64decode(payload))

def call(*args):
    result = subprocess.run([sys.executable, *args], cwd=root, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            check=True, timeout=1800)
    print(result.stdout[-3500:])
    return result.stdout

def save_text(name, value):
    (root/name).write_text(value, encoding='utf8')

save_text('F2_Acceptance_Test_Log.txt', call('-m', 'unittest', '-v', 'test_f1','test_f2'))
save_text('F2_Local_Run_Log.txt', call('ace_f2.py', 'run','--root','evidence'))
save_text('F2_Verification_Log.txt', call('ace_f2.py', 'verify','--root','evidence'))
save_text('F2_Summary_JSON.txt', call('ace_f2.py', 'summary','--root','evidence'))
metrics = json.loads((root/'evidence'/'reports'/'metrics.json').read_text())
a = sum(v['pass'] for k,v in metrics['metrics'].items() if k.startswith('A_CONTROL'))
b = sum(v['pass'] for k,v in metrics['metrics'].items() if k.startswith('B_ADAPT'))
assert (a,b)==(79,94), f'Unexpected task scores: A={a}, B={b}'
assert metrics['heldout_attempts']==0 and metrics['additional_cost_idr']==0
assert all(x['complete'] for x in metrics['runs'])
files=sorted(p for p in root.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt')
manifest=''.join(hashlib.sha256(p.read_bytes()).hexdigest()+'  '+p.relative_to(root).as_posix()+'\n' for p in files)
(root/'SHA256SUMS.txt').write_text(manifest, encoding='utf8')
archive = root/'ACE_F2_reproduced_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(root.rglob('*')):
        if p.is_file() and p!=archive:
            z.write(p,arcname=p.relative_to(root).as_posix())
assert zipfile.ZipFile(archive).testzip() is None
print('FINAL RESULT: PASS | A=',a,'/96 | B=',b,'/96 | 30/30 tests | 6/6 runs | held-out NOT_RUN')
print('EVIDENCE:',archive)
try:
    from google.colab import files as colab_files
    colab_files.download(str(archive))
except ImportError:
    print('If outside Colab, copy the archive from:',archive)
